In [ ]:
from pathlib import Path
import os
import sys
import json

REPO_ROOT = (
    Path.cwd() if (Path.cwd() / "diffusion_project").is_dir() else Path.cwd().parent
)
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
# Edit this cell, then Restart & Run All.
from pathlib import Path
import os

TOPDIR = Path(os.environ.get("EXPERIMENT_ROOT", "./results"))
DATASETS = ["celeba64", "celeba32", "fashion_mnist", "cifar10", "mnist"]
MODEL_TYPES = ["dit", "unet"]
RUN_SUFFIXES = ["", "_b", "_c"]
SUBSETS = [1, 2]
SUBSET_SIZE = 10000
CHECKPOINT_ROOT = TOPDIR / "model_checkpoints"
CHECKPOINT_NAME_TEMPLATE = "{kind}_10k_s{subset}{suffix}"
# Match this template to the configured training subset size.
CHECKPOINT_DIR_OVERRIDES = {}  # {(dataset, kind, subset, suffix): Path('/...')}

SAVED_TENSORS_ROOT = TOPDIR / "saved_tensors"
SEEDS_FILENAME = "seeds.pt"
# Choose a shared reference or one reference per training subset.
REFERENCE_MODE = "shared"  # 'shared' or 'per_subset'
SHARED_REFERENCE_FILENAME = "subset1.pt"
SUBSET_REFERENCE_TEMPLATE = "subset{subset}.pt"
# Reference selection is explicit.
REFERENCE_OVERRIDES = {}  # {(dataset, subset): Path('/...')}

EARLY_EPOCHS = list(range(1, 51))  # every epoch for the detailed zoom
LATER_EPOCHS = list(range(55, 401, 5))  # includes epoch 400
ZOOM_MAX_EPOCH = 50
AVG_K = 5  # use 1 to disable weight averaging
REQUIRE_FULL_AVERAGING_WINDOW = True  # early epochs use all available from epoch 1
NUM_SAMPLES = 100  # require this many aligned seeds/references
SAMPLING_BATCH_SIZE = 8
SAMPLE_STEPS = 100
DEVICE = os.environ.get("DEVICE", "auto")

OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", str(TOPDIR / "time_course")))
COMPUTE_TRAJECTORIES = os.environ.get("PLOT_ONLY", "0") != "1"
RESUME = True  # reuse completed, unchanged checkpoint evaluations
# Mean of per-model median correlation; mean of per-model MSE.
# Missing results stay NaN. By default, plot only epochs complete across all runs.
REQUIRE_ALL_RUNS = True
SHOW_FIGURES = os.environ.get("BATCH_MODE", "0") != "1"
FIGURE_FORMATS = ("png", "pdf")
FIGSIZE = (10.4, 7.6)
FIGURE_DPI = 200
FONT_FAMILY = "serif"
FONT_SIZE = 26
LABEL_SIZE = 30
TICK_SIZE = 24
LEGEND_SIZE = 22
LINE_WIDTH = 4.4
CORR_YLIM = (0.75, 0.95)  # None shows the full data range
MSE_YLIM = (0.02, 0.16)  # None shows the full data range
ZOOM_CORR_YLIM = (0.75, 0.95)
ZOOM_MSE_YLIM = (0.02, 0.16)
DATASET_LABELS = {
    "celeba64": "CelebA64",
    "celeba32": "CelebA32",
    "fashion_mnist": "FashionMNIST",
    "cifar10": "CIFAR10",
    "mnist": "MNIST",
}
DATASET_COLORS = {
    "celeba64": "C0",
    "fashion_mnist": "C1",
    "cifar10": "C2",
    "mnist": "C3",
    "celeba32": "C4",
}
MODEL_LABELS = {"dit": "DiT", "unet": "UNet"}
ERROR_N = 400  # Denominator for 2*sample_std/sqrt(n); actual counts are also exported.


# Time-course evaluation

Select datasets, models, subsets, and trials in the first cell. The default evaluates
both architectures on five datasets, every epoch from 1 through 50 and every fifth
epoch from 55 through 400. `AVG_K=5` averages a trailing window of checkpoint weights;
set it to 1 to evaluate individual checkpoints.

`REFERENCE_MODE='shared'` compares all runs against the selected shared analytical
reference. Choose `per_subset` for a separate reference per training subset. Reference
images must align with the saved seeds, normalization, and sampling convention.

Correlation is signed per-image Pearson correlation. Curves average per-model median
correlations; MSE averages over pixels, images, and models. Undefined correlations
remain NaN. The error columns use pooled per-image sample standard deviations and
the configurable denominator `ERROR_N`; see [methods](../docs/methods.md).

Each completed model/epoch evaluation is saved. Final plots are generated after the
sweep. Architecture and diffusion settings are read from checkpoint configurations.


In [ ]:
import csv
import fcntl
import hashlib
import io
import json
import math
import warnings
from contextlib import contextmanager
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Raw datasets are not needed: this analysis uses saved noise, references, and checkpoints.
EPOCHS = sorted(set(EARLY_EPOCHS + LATER_EPOCHS))
if not EPOCHS or min(EPOCHS) < 1:
    raise ValueError("Epochs must be positive")
if min(NUM_SAMPLES, SAMPLING_BATCH_SIZE, AVG_K, SAMPLE_STEPS, SUBSET_SIZE) < 1:
    raise ValueError("Sample counts, steps, subset size, and AVG_K must be positive")
if not DATASETS or not MODEL_TYPES or not RUN_SUFFIXES or not SUBSETS:
    raise ValueError("Choose at least one dataset, model type, run suffix, and subset")
if any(kind not in ("dit", "unet") for kind in MODEL_TYPES) or any(
    s not in (1, 2) for s in SUBSETS
):
    raise ValueError("Supported models: dit/unet; subsets: 1/2")
if REFERENCE_MODE not in ("shared", "per_subset"):
    raise ValueError("REFERENCE_MODE must be shared or per_subset")
RUNS = [(subset, suffix) for subset in SUBSETS for suffix in RUN_SUFFIXES]
if (
    len(set(RUNS)) != len(RUNS)
    or len(set(DATASETS)) != len(DATASETS)
    or len(set(MODEL_TYPES)) != len(MODEL_TYPES)
):
    raise ValueError("Remove duplicate datasets, model types, subsets, or run suffixes")
TASKS = [(dataset, kind) for kind in MODEL_TYPES for dataset in DATASETS]
OUTPUT_ROOT = OUTPUT_ROOT.expanduser().resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
device = torch.device(
    ("cuda" if torch.cuda.is_available() else "cpu") if DEVICE == "auto" else DEVICE
)
plt.rcParams.update(
    {
        "font.family": FONT_FAMILY,
        "font.size": FONT_SIZE,
        "axes.labelsize": LABEL_SIZE,
        "xtick.labelsize": TICK_SIZE,
        "ytick.labelsize": TICK_SIZE,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
print("Tasks:", TASKS, "| device:", device, flush=True)
print("Output:", OUTPUT_ROOT, flush=True)
print(
    f"{len(RUNS)} runs per task; {len(EPOCHS)} target epochs; {NUM_SAMPLES} paired images",
    flush=True,
)


@contextmanager
def output_lock(path):
    # Locks only analysis output; training jobs may continue writing new checkpoints.
    with open(path, "a") as handle:
        fcntl.flock(handle, fcntl.LOCK_EX)
        try:
            yield
        finally:
            fcntl.flock(handle, fcntl.LOCK_UN)


def atomic_text(path, text):
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(text)
    os.replace(temporary, path)


def checkpoint_directory(dataset, kind, subset, suffix):
    override = CHECKPOINT_DIR_OVERRIDES.get((dataset, kind, subset, suffix))
    if override is not None:
        return Path(override).expanduser().resolve()
    return (
        (
            CHECKPOINT_ROOT
            / dataset
            / CHECKPOINT_NAME_TEMPLATE.format(kind=kind, subset=subset, suffix=suffix)
        )
        .expanduser()
        .resolve()
    )


def reference_path(dataset, subset):
    override = REFERENCE_OVERRIDES.get((dataset, subset))
    if override is not None:
        return Path(override).expanduser().resolve()
    name = (
        SHARED_REFERENCE_FILENAME
        if REFERENCE_MODE == "shared"
        else SUBSET_REFERENCE_TEMPLATE.format(subset=subset)
    )
    return (SAVED_TENSORS_ROOT / dataset / name).expanduser().resolve()


def file_digest(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_image_tensor(path):
    value = torch.load(path, map_location="cpu", weights_only=False)
    if not torch.is_tensor(value) or value.ndim != 4 or len(value) < NUM_SAMPLES:
        raise ValueError(
            f"{path}: need at least {NUM_SAMPLES} images in an [N,C,H,W] tensor"
        )
    value = value[:NUM_SAMPLES].detach().float().cpu().contiguous()
    if not torch.isfinite(value).all():
        raise ValueError(f"{path}: non-finite image values")
    return value


## Network definitions and shared numerical helpers

In [ ]:
from diffusion_project.models import ConvDiT, AttentionUNet
from diffusion_project.diffusion import DiffusionSchedule


In [ ]:
def available_checkpoints(directory):
    return {
        int(path.stem[6:]): path
        for path in Path(directory).glob("epoch_*.pt")
        if path.stem[6:].isdigit()
    }


def architecture_kwargs(kind, config):
    keys = (
        (
            "image_size",
            "hidden_size",
            "patch_size",
            "depth",
            "num_heads",
            "mlp_ratio",
            "dropout",
        )
        if kind == "dit"
        else (
            "image_size",
            "model_channels",
            "channel_mult",
            "num_res_blocks",
            "attention_resolutions",
            "num_heads",
            "dropout",
        )
    )
    missing = [
        key for key in ("channels", "diffusion_steps") + keys if key not in config
    ]
    if missing:
        raise ValueError(f"{kind} checkpoint config is missing: {missing}")
    return dict(in_channels=config["channels"], **{key: config[key] for key in keys})


from diffusion_project.diffusion import ddim_sample as _ddim_sample


def ddim_sample(model, fixed_noise, schedule, sample_steps=SAMPLE_STEPS):
    return _ddim_sample(model, fixed_noise, schedule, sample_steps)


def as_numpy(value):
    return value.detach().cpu().numpy() if torch.is_tensor(value) else np.asarray(value)


def per_image_corr(a, b):
    a, b = as_numpy(a), as_numpy(b)
    if a.shape != b.shape or a.ndim != 4:
        raise ValueError("Correlation expects matching [N,C,H,W] arrays")
    a = a.reshape(len(a), -1).astype(np.float64)
    b = b.reshape(len(b), -1).astype(np.float64)
    a -= a.mean(axis=1, keepdims=True)
    b -= b.mean(axis=1, keepdims=True)
    numerator = np.sum(a * b, axis=1)
    denominator = np.sqrt(np.sum(a * a, axis=1) * np.sum(b * b, axis=1))
    result = np.full(len(a), np.nan)
    np.divide(numerator, denominator, out=result, where=denominator > 0)
    return np.clip(result, -1.0, 1.0)


def median_corr(values):
    valid = np.asarray(values)[np.isfinite(values)]
    return float(np.median(valid)) if len(valid) else float("nan")


## Compute and resume
`<OUTPUT_ROOT>/<dataset>/<model>/trajectories.npz` contains
`per_image_corr` and `per_image_mse` arrays with axes **[run, epoch, image]**,
`completed` with axes [run, epoch], and explicit `epochs`. `metadata_json` records
run ordering, parameters, source paths and SHA256 hashes; `checkpoints_json`
records the actual averaging epochs and model configuration for each evaluation.
Load with `np.load(path, allow_pickle=False)`.

Results are saved atomically after every model/epoch. Resuming checks the inputs,
settings, and checkpoint size/timestamp signatures. Missing checkpoints remain
NaN and are listed in `missing_checkpoints.json` after the sweep. Interrupted
sweeps retain all fully saved evaluations. To deliberately start a new analysis,
choose another `OUTPUT_ROOT`, or set `RESUME=False` to replace its cached data.

By default, an epoch is plotted only when all selected runs have a valid result.
Change `RUN_SUFFIXES` if you want only certain runs; `REQUIRE_ALL_RUNS=False`
permits means over available runs, with counts retained in the summaries.

In [ ]:
def averaging_epochs(files, epoch):
    if epoch not in files:
        return []
    expected = list(range(max(1, epoch - AVG_K + 1), epoch + 1))
    used = [e for e in expected if e in files]
    if REQUIRE_FULL_AVERAGING_WINDOW and used != expected:
        return []
    return used


def checkpoint_signature(files, used):
    # Detect replaced checkpoints and changed averaging windows before reusing results.
    return [[e, files[e].stat().st_size, files[e].stat().st_mtime_ns] for e in used]


def load_averaged_model(kind, dataset, files, used, image_shape):
    avg_state, config = None, None
    for epoch in used:
        checkpoint = torch.load(files[epoch], map_location="cpu", weights_only=False)
        current = checkpoint["config"]
        if int(checkpoint["epoch"]) != epoch:
            raise ValueError(f"{files[epoch]}: saved epoch disagrees with filename")
        kwargs = architecture_kwargs(kind, current)
        if config is not None and (
            kwargs != architecture_kwargs(kind, config)
            or current["diffusion_steps"] != config["diffusion_steps"]
        ):
            raise ValueError(
                f"{files[epoch]}: incompatible configurations in averaging window"
            )
        if "data_config" in checkpoint:
            data = checkpoint["data_config"]
            saved_dataset = (
                "celeba32" if data["dataset"] == "celeba" else data["dataset"]
            )
            if saved_dataset != dataset or data["subset_size"] != SUBSET_SIZE:
                raise ValueError(f"{files[epoch]}: dataset/subset-size mismatch")
        config = dict(current)
        state = checkpoint["model_state_dict"]
        if avg_state is None:
            avg_state = {key: value.clone() for key, value in state.items()}
        else:
            if state.keys() != avg_state.keys():
                raise ValueError("Checkpoint state keys differ")
            for key, value in state.items():
                if value.shape != avg_state[key].shape:
                    raise ValueError(f"Checkpoint shape mismatch: {key}")
                if torch.is_floating_point(value):
                    avg_state[key].add_(value)
                else:
                    avg_state[key] = value.clone()
        del checkpoint, state
    for value in avg_state.values():
        if torch.is_floating_point(value):
            value.div_(len(used))
    expected = (config["channels"], config["image_size"], config["image_size"])
    if tuple(image_shape) != expected:
        raise ValueError(
            f"Checkpoint expects {expected}; saved noise has {tuple(image_shape)}"
        )
    model = (ConvDiT if kind == "dit" else AttentionUNet)(
        **architecture_kwargs(kind, config)
    )
    model.load_state_dict(avg_state, strict=True)
    model.to(device).eval()
    return model, config


def analysis_settings():
    return {
        "analysis_version": 2,
        "epochs": EPOCHS,
        "runs": [[s, r] for s, r in RUNS],
        "subset_size": SUBSET_SIZE,
        "num_samples": NUM_SAMPLES,
        "sample_steps": SAMPLE_STEPS,
        "avg_k": AVG_K,
        "require_full_averaging_window": REQUIRE_FULL_AVERAGING_WINDOW,
        "reference_mode": REFERENCE_MODE,
        "shared_reference_filename": SHARED_REFERENCE_FILENAME,
        "subset_reference_template": SUBSET_REFERENCE_TEMPLATE,
    }


def save_progress(path, data):
    temporary = path.with_name(path.name + ".tmp")
    with open(temporary, "wb") as handle:
        np.savez_compressed(
            handle,
            epochs=np.asarray(EPOCHS),
            per_image_corr=data["corr"],
            per_image_mse=data["mse"],
            completed=data["completed"],
            metadata_json=json.dumps(data["metadata"], sort_keys=True),
            checkpoints_json=json.dumps(data["checkpoints"], sort_keys=True),
        )
    os.replace(temporary, path)


def read_progress(path):
    with np.load(path, allow_pickle=False) as saved:
        return {
            "corr": saved["per_image_corr"].copy(),
            "mse": saved["per_image_mse"].copy(),
            "completed": saved["completed"].copy(),
            "metadata": json.loads(str(saved["metadata_json"].item())),
            "checkpoints": json.loads(str(saved["checkpoints_json"].item())),
        }


def run_trajectory(dataset, kind):
    pair_dir = OUTPUT_ROOT / dataset / kind
    pair_dir.mkdir(parents=True, exist_ok=True)
    cache_path = pair_dir / "trajectories.npz"
    with output_lock(pair_dir / ".compute.lock"):
        noise_path = (
            (SAVED_TENSORS_ROOT / dataset / SEEDS_FILENAME).expanduser().resolve()
        )
        seeds = load_image_tensor(noise_path)
        ref_paths = {s: reference_path(dataset, s) for s in SUBSETS}
        unique_refs = {p: load_image_tensor(p) for p in set(ref_paths.values())}
        refs = {s: unique_refs[p] for s, p in ref_paths.items()}
        if any(ref.shape != seeds.shape for ref in refs.values()):
            raise ValueError(f"{dataset}: saved seed/reference shapes differ")
        directories = [checkpoint_directory(dataset, kind, s, r) for s, r in RUNS]
        metadata = {
            "dataset": dataset,
            "model_type": kind,
            "settings": analysis_settings(),
            "seed_file": str(noise_path),
            "seed_sha256": file_digest(noise_path),
            "references": {
                str(s): {"path": str(p), "sha256": file_digest(p)}
                for s, p in ref_paths.items()
            },
            "checkpoint_dirs": [str(p) for p in directories],
            "torch_version": str(torch.__version__),
        }
        shape = (len(RUNS), len(EPOCHS), NUM_SAMPLES)
        if RESUME and cache_path.exists():
            data = read_progress(cache_path)
            # Torch versions are recorded for provenance but do not invalidate metrics.
            old = dict(data["metadata"])
            old.pop("torch_version", None)
            new = dict(metadata)
            new.pop("torch_version", None)
            if old != new or data["corr"].shape != shape:
                raise ValueError(
                    f"{cache_path}: analysis inputs/settings changed; choose another OUTPUT_ROOT or set RESUME=False"
                )
        else:
            data = {
                "corr": np.full(shape, np.nan),
                "mse": np.full(shape, np.nan),
                "completed": np.zeros(shape[:2], dtype=bool),
                "metadata": metadata,
                "checkpoints": {},
            }
            save_progress(cache_path, data)
        missing = []
        resumed = 0
        for run_index, ((subset, suffix), directory) in enumerate(
            zip(RUNS, directories)
        ):
            files = available_checkpoints(directory)  # snapshot once per trained model
            if not files:
                print(f"MISSING checkpoint directory: {directory}", flush=True)
            for epoch_index, epoch in enumerate(EPOCHS):
                used = averaging_epochs(files, epoch)
                key = f"{run_index}:{epoch}"
                if not used:
                    missing.append(
                        {
                            "subset": subset,
                            "suffix": suffix,
                            "epoch": epoch,
                            "reason": "missing target or incomplete averaging window",
                        }
                    )
                    # Do not keep a result when the checkpoint/window is no longer available.
                    data["corr"][run_index, epoch_index] = np.nan
                    data["mse"][run_index, epoch_index] = np.nan
                    data["completed"][run_index, epoch_index] = False
                    data["checkpoints"].pop(key, None)
                    continue
                signature = checkpoint_signature(files, used)
                if (
                    data["completed"][run_index, epoch_index]
                    and data["checkpoints"].get(key, {}).get("signature") == signature
                ):
                    resumed += 1
                    continue
                model, config = load_averaged_model(
                    kind, dataset, files, used, seeds.shape[1:]
                )
                schedule = DiffusionSchedule(config["diffusion_steps"], device)
                corr_parts, mse_parts = [], []
                try:
                    for start in range(0, NUM_SAMPLES, SAMPLING_BATCH_SIZE):
                        stop = min(start + SAMPLING_BATCH_SIZE, NUM_SAMPLES)
                        sample = ddim_sample(model, seeds[start:stop], schedule)
                        reference = refs[subset][start:stop]
                        if not torch.isfinite(sample).all():
                            raise ValueError(
                                f"{directory}, epoch {epoch}: non-finite samples"
                            )
                        corr_parts.append(per_image_corr(sample, reference))
                        mse_parts.append(
                            np.mean(
                                (
                                    as_numpy(sample).astype(np.float64)
                                    - as_numpy(reference)
                                )
                                ** 2,
                                axis=(1, 2, 3),
                            )
                        )
                finally:
                    del model, schedule
                if checkpoint_signature(files, used) != signature:
                    raise RuntimeError(
                        f"{directory}: a checkpoint changed during evaluation; rerun to retry"
                    )
                corr, mse = np.concatenate(corr_parts), np.concatenate(mse_parts)
                data["corr"][run_index, epoch_index] = corr
                data["mse"][run_index, epoch_index] = mse
                data["completed"][run_index, epoch_index] = True
                data["checkpoints"][key] = {
                    "signature": signature,
                    "averaged_epochs": used,
                    "config": config,
                }
                save_progress(
                    cache_path, data
                )  # persist each model/epoch before continuing
                print(
                    f'{dataset}/{kind} s{subset}{suffix or "_a"} epoch {epoch}: '
                    f"median corr.={median_corr(corr):.5f}; MSE={mse.mean():.6g}; "
                    f"valid corr.={np.isfinite(corr).sum()}/{NUM_SAMPLES}",
                    flush=True,
                )
        save_progress(cache_path, data)
        atomic_text(
            pair_dir / "missing_checkpoints.json", json.dumps(missing, indent=2) + "\n"
        )
        print(
            f'{dataset}/{kind}: {data["completed"].sum()}/{data["completed"].size} evaluations saved; '
            f"{resumed} reused; {len(missing)} missing",
            flush=True,
        )


## Final exports and plotting
Each dataset/model directory receives `per_run_metrics.csv` and `summary.csv`.
`plots/` receives corr. and MSE curves over the full epoch range and the first
50 epochs, for DiT and UNet separately, as PNG and PDF files (eight plots in each
format). `completion_status.json` reports which tasks/epochs are complete.

Plots are built from saved results. You can regenerate them without GPU
inference by setting
`COMPUTE_TRAJECTORIES=False` and running the notebook. There are no plots inside
the epoch loop, and batch execution never modifies this source notebook.

In [ ]:
def csv_text(rows, fields):
    stream = io.StringIO()
    writer = csv.DictWriter(stream, fieldnames=fields)
    writer.writeheader()
    writer.writerows(rows)
    return stream.getvalue()


def finite_mean(values):
    values = np.asarray(values)
    valid = values[np.isfinite(values)]
    return float(valid.mean()) if len(valid) else float("nan")


def finite_std(values):
    values = np.asarray(values)
    valid = values[np.isfinite(values)]
    return float(valid.std(ddof=1)) if len(valid) > 1 else float("nan")


def summarize(data):
    corr, mse, done = data["corr"], data["mse"], data["completed"]
    run_median = np.full(done.shape, np.nan)
    run_mse = np.full(done.shape, np.nan)
    run_rows = []
    for i, (subset, suffix) in enumerate(RUNS):
        for j, epoch in enumerate(EPOCHS):
            if done[i, j]:
                run_median[i, j] = median_corr(corr[i, j])
                run_mse[i, j] = finite_mean(mse[i, j])
            run_rows.append(
                {
                    "subset": subset,
                    "run_suffix": suffix,
                    "epoch": epoch,
                    "completed": bool(done[i, j]),
                    "median_corr": run_median[i, j],
                    "mean_mse": run_mse[i, j],
                    "valid_corr_images": (
                        int(np.isfinite(corr[i, j]).sum()) if done[i, j] else 0
                    ),
                    "corr_std_images": (
                        finite_std(corr[i, j]) if done[i, j] else float("nan")
                    ),
                    "mse_std_images": (
                        finite_std(mse[i, j]) if done[i, j] else float("nan")
                    ),
                }
            )
    rows = []
    for j, epoch in enumerate(EPOCHS):
        rows.append(
            {
                "epoch": epoch,
                "mean_median_corr": finite_mean(run_median[:, j]),
                "mean_mse": finite_mean(run_mse[:, j]),
                "corr_2sigma_sqrt_n": 2
                * finite_std(corr[done[:, j], j, :])
                / np.sqrt(ERROR_N),
                "mse_2sigma_sqrt_n": 2
                * finite_std(mse[done[:, j], j, :])
                / np.sqrt(ERROR_N),
                "error_n": ERROR_N,
                "n_corr_images": int(np.isfinite(corr[done[:, j], j, :]).sum()),
                "n_mse_images": int(np.isfinite(mse[done[:, j], j, :]).sum()),
                "std_run_median_corr": finite_std(run_median[:, j]),
                "std_run_mean_mse": finite_std(run_mse[:, j]),
                "n_corr_runs": int(np.isfinite(run_median[:, j]).sum()),
                "n_mse_runs": int(np.isfinite(run_mse[:, j]).sum()),
                "n_completed_runs": int(done[:, j].sum()),
                "n_requested_runs": len(RUNS),
            }
        )
    return run_rows, rows


def export_and_plot():
    # Array tasks share this short lock so the last task produces the final combined plots.
    # Atomic .npz writes allow consistent reads while another task is still computing.
    with output_lock(OUTPUT_ROOT / ".plots.lock"):
        summaries, status = {}, []
        for dataset, kind in TASKS:
            pair_dir = OUTPUT_ROOT / dataset / kind
            path = pair_dir / "trajectories.npz"
            if not path.exists():
                status.append(
                    {
                        "dataset": dataset,
                        "model_type": kind,
                        "status": "not yet computed",
                    }
                )
                continue
            data = read_progress(path)
            if data["metadata"]["settings"] != analysis_settings():
                warnings.warn(
                    f"{path}: different analysis settings; excluded from these plots"
                )
                status.append(
                    {
                        "dataset": dataset,
                        "model_type": kind,
                        "status": "different settings",
                    }
                )
                continue
            run_rows, rows = summarize(data)
            atomic_text(
                pair_dir / "per_run_metrics.csv", csv_text(run_rows, list(run_rows[0]))
            )
            atomic_text(pair_dir / "summary.csv", csv_text(rows, list(rows[0])))
            summaries[dataset, kind] = rows
            status.append(
                {
                    "dataset": dataset,
                    "model_type": kind,
                    "completed_evaluations": int(data["completed"].sum()),
                    "requested_evaluations": int(data["completed"].size),
                    "complete_epochs": [
                        r["epoch"] for r in rows if r["n_completed_runs"] == len(RUNS)
                    ],
                }
            )
        atomic_text(
            OUTPUT_ROOT / "completion_status.json", json.dumps(status, indent=2) + "\n"
        )
        plot_dir = OUTPUT_ROOT / "plots"
        plot_dir.mkdir(exist_ok=True)
        for kind in MODEL_TYPES:
            for metric, label, count_key, limits, zoom_limits in (
                (
                    "mean_median_corr",
                    "Mean median corr.",
                    "n_corr_runs",
                    CORR_YLIM,
                    ZOOM_CORR_YLIM,
                ),
                ("mean_mse", "MSE", "n_mse_runs", MSE_YLIM, ZOOM_MSE_YLIM),
            ):
                for zoom in (False, True):
                    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=FIGURE_DPI)
                    plotted = False
                    for dataset in DATASETS:
                        rows = summaries.get((dataset, kind))
                        if rows is None:
                            continue
                        chosen = [
                            r for r in rows if not zoom or r["epoch"] <= ZOOM_MAX_EPOCH
                        ]
                        values = [
                            (
                                r[metric]
                                if not REQUIRE_ALL_RUNS or r[count_key] == len(RUNS)
                                else np.nan
                            )
                            for r in chosen
                        ]
                        if not np.isfinite(values).any():
                            continue
                        ax.plot(
                            [r["epoch"] for r in chosen],
                            values,
                            linewidth=LINE_WIDTH,
                            color=DATASET_COLORS.get(dataset),
                            label=DATASET_LABELS.get(dataset, dataset),
                        )
                        plotted = True
                    ax.set_xlabel("Epoch")
                    ax.set_ylabel(label)
                    ax.set_title(
                        MODEL_LABELS[kind]
                        + (f" (first {ZOOM_MAX_EPOCH} epochs)" if zoom else "")
                    )
                    ax.grid(alpha=0.25, linewidth=0.8)
                    if zoom:
                        ax.set_xlim(0, ZOOM_MAX_EPOCH)
                    ylim = zoom_limits if zoom else limits
                    if ylim is not None:
                        ax.set_ylim(*ylim)
                    if plotted:
                        ax.legend(
                            fontsize=LEGEND_SIZE,
                            loc=(
                                "lower right"
                                if metric == "mean_median_corr"
                                else "upper right"
                            ),
                        )
                    else:
                        ax.text(
                            0.5,
                            0.5,
                            (
                                "No complete results yet"
                                if REQUIRE_ALL_RUNS
                                else "No results yet"
                            ),
                            transform=ax.transAxes,
                            ha="center",
                            va="center",
                            fontsize=16,
                        )
                    fig.tight_layout()
                    name = f'{kind}_{"corr" if metric == "mean_median_corr" else "mse"}_{"first" + str(ZOOM_MAX_EPOCH) if zoom else "full"}'
                    for extension in FIGURE_FORMATS:
                        fig.savefig(plot_dir / f"{name}.{extension}", dpi=FIGURE_DPI)
                    if SHOW_FIGURES:
                        plt.show()
                    plt.close(fig)
        print("Final data/plots updated:", OUTPUT_ROOT, flush=True)
        for item in status:
            print(
                item["dataset"],
                item["model_type"],
                (
                    f"{item['completed_evaluations']}/{item['requested_evaluations']} evaluations"
                    if "completed_evaluations" in item
                    else item["status"]
                ),
                flush=True,
            )


In [ ]:
try:
    if COMPUTE_TRAJECTORIES:
        for dataset, kind in TASKS:
            run_trajectory(dataset, kind)
finally:
    # Export completed results even if a later evaluation raises an exception.
    export_and_plot()
